Disable PyTorch's strict safety checks entirely

In [ ]:
import os
# Critical: Setting this BEFORE importing torch/ultralytics
os.environ['TORCH_FORCE_WEIGHTS_ONLY_LOAD'] = '0'

Cleanup Section

In [ ]:
import shutil
from pathlib import Path
# Clean up previous runs for a fresh start
if Path("runs/detect/train").exists():
    shutil.rmtree("runs/detect/train", ignore_errors=True)
if Path("../data/yolo_dataset").exists():
    shutil.rmtree("../data/yolo_dataset", ignore_errors=True)
print("Setup complete. Previous runs cleaned up.")

Setup and Imports

In [ ]:
from pathlib import Path

# Install necessary libraries
%pip install ultralytics==8.0.196 scikit-learn datasets pyyaml opencv-python pillow matplotlib

# Import modules
from lucky_data_prep import load_and_merge_datasets, MaskToBBoxConverter 
from lucky_yolo_trainer import CustomYOLOTrainer

# --- IMPROVED Configuration ---
DATA_YAML_PATH = "../data/yolo_dataset/data.yaml"

# Used nano model
MODEL_WEIGHTS = "yolov8n.pt"

Step 1 - Load and Merge Data

In [ ]:
dataset = load_and_merge_datasets()

if dataset is None:
    print("Pipeline stopped: Failed to load dataset.")

Step 2 - Convert Data to YOLO Format

In [ ]:
if dataset:
    converter = MaskToBBoxConverter()
    stats = converter.convert_dataset(dataset)
    
    if stats:
        print("\nDataset ready for YOLO training.")
    else:
        print("Pipeline stopped: Failed to convert dataset.")

Diagnostics

In [ ]:
import os
import yaml

# Check the path set in Cell 1
DATA_YAML_PATH = "../data/yolo_dataset/data.yaml"

if os.path.exists(DATA_YAML_PATH):
    print(f"--- Content of {DATA_YAML_PATH} ---")
    with open(DATA_YAML_PATH, 'r') as f:
        data_content = yaml.safe_load(f)
        print(yaml.dump(data_content, indent=4))
    
    # Check if the images/labels exist relative to the 'path'
    base_path = data_content.get('path', 'NOT_FOUND')
    train_images = os.path.join(base_path, data_content.get('train', 'images/train'))
    
    print(f"\n--- Checking Internal Paths ---")
    print(f"Base Path: {base_path}")
    print(f"Train Images Path: {train_images}")
    
    if os.path.exists(train_images):
        print(f"Train images directory found.")
        print(f"Number of files in train/images: {len(os.listdir(train_images))}")
    else:
        print(f"Train images directory NOT FOUND. Check your 'path' in data.yaml.")

else:
    print(f"ERROR: data.yaml not found at {DATA_YAML_PATH}")

Step 3 - Load Model and Freeze Layers

In [ ]:
# Initialize trainer
trainer = CustomYOLOTrainer(model_size=MODEL_WEIGHTS, data_yaml=DATA_YAML_PATH)

# Load base model (uses the PyTorch fix)
if trainer.load_model():
    # Freeze backbone layers (a key step for better fine-tuning)
    trainer.freeze_backbone(num_layers=10)

Step 4 - Fine-Tune the Model

In [ ]:
if trainer.model:
    print("\n" + "=" * 60)
    print(f"STARTING FINE-TUNING on {MODEL_WEIGHTS} (50 epochs)")
    print("=" * 60)
    
    # **Run the training:**
    results = trainer.train(epochs=50, batch_size=8, patience=10) 
    
    if results:
        print(f"Training finished. Best model saved to: {results.save_dir}/weights/best.pt")
        # Save the path to the best model for the validation step
        best_weights_path = Path(results.save_dir) / "weights" / "best.pt"
    else:
        best_weights_path = None
        print("Training failed. Validation skipped.")
else:
    best_weights_path = None


Step 5 - Final Validation

In [ ]:
if best_weights_path and best_weights_path.exists(): 
    print(f"\n" + "=" * 60)
    print("STEP 5: LOADING BEST MODEL AND VALIDATING")
    print("=" * 60)
    
    # Load the best weights from the training run
    final_trainer = CustomYOLOTrainer(model_size=str(best_weights_path), data_yaml=DATA_YAML_PATH)
    final_trainer.load_model()
    
    # **Run final validation:**
    if final_trainer.model:
        metrics = final_trainer.validate()
        
        if metrics:
            print("\nFinal Model Performance is in the metrics above.")
else:
    print("\nFinal Validation Skipped: No best model weights found.")
